<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_07_model_training/stage_07_00_model_training_seq2one_plan.ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07 – Model Training (Seq2One) – Objetivo y Modelos**


## **1. Objetivo de predicción (definición formal)**


En este stage se entrenan modelos **seq2one** para predecir un valor escalar futuro a partir de una ventana histórica intradía.

- **Entrada (X)**

  Ventana histórica intradía de tamaño **60 × 20**:

  $$
  X \in \mathbb{R}^{60 \times 20}
  $$

  donde:
  - $60$ representa minutos consecutivos (ventana temporal),
  - $20$ representa la cantidad de features por minuto.

  En entrenamiento, el dataset adopta la forma:

  $$
  X \in \mathbb{R}^{N \times 60 \times 20}
  $$

  donde $N$ es el número total de ventanas.

- **Salida / Target (Y)**

  Un único valor escalar asociado a cada ventana:

  $$
  Y \in \mathbb{R}^{1}
  $$

  que representa la variable objetivo futura definida en los stages previos.

- **Interpretación**

  El modelo aprende una función:

  $$
  f: \mathbb{R}^{60 \times 20} \rightarrow \mathbb{R}
  $$

  y predice **un único valor final** por ventana histórica.


### **1.1. Qué se evalúa en este stage_07**

En el **stage_07 no se realiza la comparación final entre modelos** ni se selecciona el “mejor modelo”.

Este stage garantiza que:

- cada modelo es entrenado correctamente,
- se respetan los splits temporales definidos,
- la complejidad y regularización están fijadas *antes* del entrenamiento,
- el proceso es reproducible y comparable.

El análisis comparativo y la evaluación definitiva se realizan recién en el **stage_08**.

> Nota: durante este stage puede utilizarse el set de *validation* únicamente para control de entrenamiento (early stopping, chequeo de convergencia), pero **no para decidir modelos**.


### **1.2. Qué NO es este problema**


Quedan explícitamente fuera de alcance:

- **Seq2seq**: predicción de trayectorias completas de múltiples pasos futuros.
- **Targets agregados ad-hoc**: suma, promedio o retornos acumulados calculados fuera del target definido.
- **Múltiples regresiones independientes por paso futuro** (heads separados sin estructura temporal compartida).



## **2. Entrenamiento de múltiples modelos**

Se entrenan múltiples modelos bajo reglas estrictamente comunes:

- mismo dataset,
- mismo split temporal (train / valid / test),
- mismo escalamiento,
- misma definición del target,
- misma función objetivo.

Ejemplos típicos (alineados al enfoque del libro):

- Naive (baseline),
- modelos lineales (Ridge, Lasso),
- árboles y ensembles,
- redes neuronales (MLP, LSTM, etc.).

Cada modelo:

- se entrena utilizando **exclusivamente el set de entrenamiento**,
- con hiperparámetros y regularización definidos previamente,
- sin ajustes posteriores basados en métricas de test.

## **3. Modelos a evaluar (comparación escalonada)**


Se define un baseline obligatorio para establecer un piso mínimo de desempeño
y detectar rápidamente sobreajuste o complejidad innecesaria.

### **3.1. Modelos propuestos**

#### **(A) Baselines obligatorios**


1. **Naive / Persistence**

    - Predice el último valor observado de la ventana.
    - Define el piso mínimo de performance.
    - Modelo de control.
    - Notebook: `stage_07_01_naive_seq2one.ipynb`


#### **(B) Modelos clásicos**


2. **Ridge / Lasso**
    - Modelos lineales sobre la ventana aplanada (60x20 → 1200).
    - Referencia interpretable y estable.
    - Notebook: `stage_07_02_linear_seq2one.ipynb`

3. **MLP (Feedforward)**
    - Entrada: ventana aplanada.
    - Salida: escalar.
    - Primera referencia no lineal.
    - Notebook: `stage_07_03_mlp_seq2one.ipynb`



#### **(C) Modelos temporales (many-to-one)**


4. **LSTM many-to-one**

    - Encoder recurrente.
    - Se utiliza únicamente el último estado oculto.
    - Salida escalar.
    - Notebook: `stage_07_04_lstm_seq2one.ipynb`

5. **GRU many-to-one**

    - Variante más simple y estable que LSTM.
    - Notebook: `stage_07_05_gru_seq2one.ipynb`

#### **(D) Modelo no recurrente robusto**


6. **TCN many-to-one**

    - Convoluciones causales dilatadas.
    - Último timestep → proyección escalar.
    - Muy buena estabilidad intradía.
    - Notebook: `stage_07_06_tcn_seq2one.ipynb`


#### **(E)  Modelo con atención**


7. **Transformer encoder-only (seq2one)**

    - Solo encoder.
    - Pooling o token final → salida escalar.
    - Mayor capacidad, mayor riesgo.
    - Notebook: `stage_07_07_transformer_seq2one.ipynb`

### **3.2. Regularización de modelos**

La regularización forma parte del diseño de cada modelo y es clave para controlar
la capacidad y garantizar generalización.  
Cada familia de modelos requiere un esquema distinto.

#### **(A) Baselines obligatorios**


1. **Naive / Persistence**

    **Regularización:** No aplica.

    - No tiene parámetros entrenables.
    - No puede sobreajustar en sentido de Machine Learning.
    - Rol: define el piso mínimo de performance.
    - Si un modelo entrenable no supera este baseline en validation, se descarta.

#### **(B) Modelos clásicos**


2. **Ridge / Lasso**

    **Riesgo:** Bajo, controlado por diseño.

    **Regularización recomendada:**

    - Penalización L2 (Ridge) o L1 (Lasso) como mecanismo principal.
    - Selección del coeficiente de regularización previa al entrenamiento.
    - No requiere dropout ni early stopping.

    **Rol:** referencia lineal, estable e interpretable.

3. **MLP (Feedforward, seq2one)**

    **Riesgo principal:** Sobreajuste por capacidad del modelo.

    **Regularización recomendada:**

    - Penalización L2 (weight decay) como mecanismo principal.
    - Early stopping monitoreando la pérdida en validation.
    - Arquitectura limitada:
      - pocas capas (1–2),
      - número reducido de neuronas.
    - Dropout leve (opcional, solo si se observa sobreajuste claro).

    **Interpretación:** modelo no lineal flexible que requiere regularización explícita.


#### **(C) Modelos temporales (many-to-one)**


4. **LSTM many-to-one**

    **Riesgo:** Alta capacidad combinada con memoria de largo plazo.

    **Regularización recomendada:**

    - Early stopping obligatorio.
    - Tamaño moderado del hidden state.
    - Número de capas limitado (1–2).
    - Dropout en entradas y salidas (no recurrente).
    - Penalización L2 suave (opcional).

    **Nota:** se utiliza únicamente el último estado oculto para la predicción escalar.


5. **GRU many-to-one**

    **Riesgo:** Menor que LSTM, pero presente.

    **Regularización recomendada:**

    - Early stopping como mecanismo principal.
    - Limitar dimensión del hidden state.
    - Limitar número de capas.
    - Dropout opcional entre capas (no dentro de la recurrencia).

    **Ventaja:** mayor estabilidad y menor necesidad de regularización que LSTM.


#### **(D) Modelo no recurrente robusto**


6. **TCN (Temporal Convolutional Network, seq2one)**

    **Riesgo:** Campo receptivo excesivo o filtros redundantes.

    **Regularización recomendada:**

    - Limitar profundidad del modelo (dilataciones).
    - Limitar número de filtros por capa.
    - Dropout (especialmente efectivo en TCN).
    - Early stopping.
    - Weight normalization (si está disponible).

    **Ventaja clave:** buena generalización intradía con menor complejidad recurrente.


#### **(E) Modelo con atención**


7. **Transformer encoder-only (seq2one)**

    **Riesgo:** Sobreajuste por alta capacidad.

    **Regularización obligatoria:**

    - Dropout en bloques de atención y feed-forward.
    - Early stopping estricto.
    - Limitar número de capas del encoder.
    - Limitar dimensión del embedding.
    - Pooling simple o token final para salida escalar.

    **Regla práctica:** sin regularización fuerte, el modelo no generaliza.

#### **Resumen operativo**


- **Naive:** sin regularización.
- **Ridge / Lasso:** regularización integrada (L1 / L2).
- **MLP:** L2 + early stopping (+ dropout opcional).
- **GRU / LSTM:** early stopping + tamaño controlado + dropout.
- **TCN:** control de profundidad + dropout.
- **Transformer encoder:** dropout fuerte + límites estrictos.

La regularización no es un agregado opcional:  
define la capacidad efectiva del modelo y su comportamiento fuera de muestra.

La comparación final entre modelos se realiza recién en el **stage_08**.

## **4. Definición de la comparación Predicción vs Valor Real (Seq2One)**


En este stage, la evaluación del modelo se realiza bajo un esquema
**seq2one**, donde cada ventana histórica produce una única predicción escalar.


### **4.1 Esquema de predicción**

Para cada muestra $t$, el modelo recibe como entrada una ventana histórica:

$$
X_t \in \mathbb{R}^{60 \times 20}
$$

correspondiente a 60 minutos consecutivos con 20 features por minuto.

A partir de esta entrada, el modelo produce una única predicción escalar:

$$
\hat{y}_t \in \mathbb{R}
$$

que representa el valor futuro del target definido (por ejemplo,  
$\Delta pts_{t,h}$ para un horizonte fijo $h$).



### **4.2 Comparación contra el valor real**


La predicción se compara directamente contra el valor real observado:

$$
y_t \in \mathbb{R}
$$

La comparación es **escalar contra escalar**, sin:

- generar trayectorias,
- aplicar agregaciones intermedias,
- ni comparar secuencias completas.

Cada ventana histórica tiene **una única predicción** y **un único valor real asociado**.

### **4.3 Cálculo de métricas**

A partir de la comparación $\hat{y}_t$ vs $y_t$, las métricas se calculan
sobre el conjunto de muestras:

- **Métricas de Machine Learning**
  - MAE
  - RMSE
  - R² (opcional)
  - Métricas direccionales (signo de la predicción vs signo real)

Las métricas se agregan sobre todas las ventanas del split correspondiente.


### **4.4 Uso de los splits (criterio de evaluación)**


Siguiendo el criterio operativo:

- **TRAIN**: utilizado exclusivamente para el aprendizaje del modelo.
- **VALID**: utilizado para medir desempeño y descartar modelos que no generalizan.
- **TEST**: utilizado únicamente una vez finalizada la selección del modelo.

No se ajustan hiperparámetros en función del set de test.

### **4.5. Derivación posterior de métricas económicas**


A partir de la predicción escalar y del valor real observado, pueden derivarse
posteriormente métricas económicas, tales como:

- expectativa de valor (EV),
- métricas TP / SL,
- drawdown simulado,

utilizando el **movimiento real observado** en el horizonte futuro.

Estas métricas **no forman parte del entrenamiento**, sino del análisis
posterior (`stage_08`).

### **Nota metodológica**


En este stage el problema **no es seq2seq**.
No se comparan trayectorias ni secuencias completas.

El objetivo es evaluar la capacidad del modelo para predecir correctamente
un **valor escalar futuro**, bajo un esquema controlado y reproducible.

## **5. Métricas de predicción (Machine Learning)**

Dado que el problema consiste en la predicción de un **valor escalar futuro**
bajo un esquema **seq2one**, la evaluación del desempeño del modelo se realiza
mediante un conjunto reducido de métricas, priorizando comparabilidad,
estabilidad y alineación con el objetivo del proyecto.

Las métricas se calculan siempre **fuera de muestra (validation)**.
El conjunto **test** se reserva exclusivamente para la evaluación final,
una vez seleccionado el modelo.

### **5.1 Métricas principales de error (seq2one)**

Para cada muestra $t$, la predicción escalar $\hat{y}_t$ se compara
directamente contra el valor real observado $y_t$.

Las métricas principales se calculan agregando el error sobre todas las
muestras del conjunto evaluado.

**1. MAE (Mean Absolute Error)**

Error absoluto medio entre predicción y valor real:

$$
\text{MAE} = \frac{1}{N} \sum_{t=1}^{N} |\hat{y}_t - y_t|
$$

**2. RMSE (Root Mean Squared Error)**

Raíz del error cuadrático medio:

$$
\text{RMSE} = \sqrt{\frac{1}{N} \sum_{t=1}^{N} (\hat{y}_t - y_t)^2}
$$

Estas métricas constituyen el **criterio principal de comparación y ranking**
de modelos.


### **5.2 Métrica direccional (complementaria)**


Como métrica complementaria se utiliza la **Directional Accuracy (DA)**,
definida como la proporción de casos en los que el signo de la predicción
coincide con el signo del valor real:

$$
\text{DA} = \mathbb{P}[\text{sign}(\hat{y}_t) = \text{sign}(y_t)]
$$

Esta métrica:

- no se utiliza como criterio principal de selección,
- se reporta con fines interpretativos,
- conecta la predicción con la dirección esperada del movimiento.


### **5.3 Coeficiente de determinación (R²)**


Se reporta adicionalmente el coeficiente de determinación $R^2$,
calculado sobre el conjunto completo de predicciones escalares.

Esta métrica se considera **descriptiva** y **complementaria**, y no se utiliza
como criterio principal de comparación debido a su limitada estabilidad
en series financieras.

### **5.4. Jerarquía de métricas**

- **Métricas principales de comparación**
  - MAE
  - RMSE

- **Métricas complementarias**
  - Directional Accuracy (DA)
  - R²

La selección y descarte de modelos se realiza **exclusivamente**
en base a las métricas principales, evaluadas sobre el conjunto de validation.


### **Nota metodológica**

En este stage no se evalúan trayectorias ni errores por horizonte.
Cada ventana histórica produce una única predicción escalar,
y la evaluación se realiza bajo un esquema estrictamente **seq2one**.

### **5.5. Función de cálculo de métricas**

In [ ]:
import numpy as np
from sklearn.metrics import r2_score

def compute_seq2one_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    *,
    compute_r2: bool = True,
    da_ignore_zeros: bool = True,
    allow_seq_inputs_take_last: bool = False,
) -> dict:
    """
    Calcula métricas simples y comparables para modelos seq2one.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    y_pred : np.ndarray
        Valores predichos con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    compute_r2 : bool
        Si True, calcula R² sobre el vector completo.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 en y_true o y_pred al calcular DA.
    allow_seq_inputs_take_last : bool
        Si True, permite inputs 2D (n_samples, seq_len) y toma el último paso [:, -1].
        Útil si algún modelo devuelve secuencia pero usted lo evalúa como many-to-one.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales.
    """

    # 1) Convertir a np.ndarray y forzar float
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    # 2) Normalizar dimensiones hacia (n_samples,)
    def _to_1d(y: np.ndarray, name: str) -> np.ndarray:
        if y.ndim == 1:
            return y
        if y.ndim == 2:
            # (n, 1) -> (n,)
            if y.shape[1] == 1:
                return y.squeeze(1)
            # (n, seq_len) -> tomar último si se permite
            if allow_seq_inputs_take_last:
                return y[:, -1]
            raise ValueError(
                f"{name} con shape {y.shape} no es válido para seq2one. "
                f"Se esperaba (n_samples,) o (n_samples, 1)."
            )
        if y.ndim == 3 and y.shape[-1] == 1:
            # (n, seq_len, 1) -> (n, seq_len) y luego último si se permite
            y2 = y.squeeze(-1)
            if allow_seq_inputs_take_last:
                return y2[:, -1]
            raise ValueError(
                f"{name} con shape {y.shape} parece seq2seq. "
                f"Active allow_seq_inputs_take_last=True si quiere tomar el último paso."
            )
        raise ValueError(
            f"{name}.ndim={y.ndim} no es válido. "
            f"Se esperaba (n,), (n,1) o (n,seq_len) si allow_seq_inputs_take_last=True."
        )

    y_true = _to_1d(y_true, "y_true")
    y_pred = _to_1d(y_pred, "y_pred")

    if y_true.shape != y_pred.shape:
        raise ValueError(
            f"y_true y y_pred deben tener el mismo shape. "
            f"Recibido y_true={y_true.shape}, y_pred={y_pred.shape}"
        )

    n_samples = int(y_true.shape[0])
    if n_samples == 0:
        raise ValueError("y_true/y_pred no pueden estar vacíos.")

    # 3) Validación numérica básica
    if not (np.isfinite(y_true).all() and np.isfinite(y_pred).all()):
        raise ValueError("Se encontraron NaN o inf en y_true/y_pred. "
                         "Limpie o enmascare antes de calcular métricas.")

    # 4) Errores
    errors = y_pred - y_true
    abs_errors = np.abs(errors)

    # 5) Métricas principales
    mae = float(abs_errors.mean())
    rmse = float(np.sqrt((errors ** 2).mean()))

    # 6) Métrica direccional (DA)
    sign_true = np.sign(y_true)
    sign_pred = np.sign(y_pred)

    if da_ignore_zeros:
        mask = (sign_true != 0) & (sign_pred != 0)
        da = float(np.mean(sign_true[mask] == sign_pred[mask])) if mask.any() else float("nan")
        da_n = int(mask.sum())
    else:
        da = float(np.mean(sign_true == sign_pred))
        da_n = n_samples

    metrics = {
        "MAE": mae,
        "RMSE": rmse,
        "DA": da,
        "DA_n": da_n,  # cuántas muestras realmente aportaron a DA (si ignore_zeros=True)
    }

    # 7) R² opcional
    if compute_r2:
        metrics["R2"] = float(r2_score(y_true, y_pred))

    return metrics


## **7. Arquitectura de notebooks**

Stage_07_XX – <MODEL_NAME> (Seq2Seq 29x8 → 29x1)

## 0) Objetivo de la notebook
- Qué modelo se entrena y por qué.
- Qué horizonte aplica (h=60 o h=90).
- Qué artefactos produce para Stage_08.

---

## 1) Setup
### 1.1 Imports
- numpy, pandas, torch/keras (según modelo)
- utilidades comunes: `compute_seq2seq_metrics`, `compute_opportunity_filter_metrics`

### 1.2 Reproducibilidad
- seeds (numpy / torch / random)
- device (cpu/cuda)
- flags de determinismo si aplica

---

## 2) Configuración (parámetros)
- `HORIZON = 60 | 90`
- `SEQ_LEN = 29`
- `N_FEATURES = 8`
- hiperparámetros del modelo
- `theta` (umbral señal) y `delta_op` (umbral oportunidad)

---

## 3) Carga de datos (inputs del pipeline)
- cargar `windows_{split}_{h}.npz` (train/valid/test)
- verificar shapes y dtypes
- sanity checks (NaN, rangos, conteos)

---

## 4) Dataloaders / batching
- dataset + dataloader
- definición clara de:
  - `X: (batch, 29, 8)`
  - `Y: (batch, 29, 1)` o `(batch, 29)`

---

## 5) Definición del modelo
- arquitectura
- función de pérdida (MSE / MAE)
- optimizer
- scheduler (opcional)

---

## 6) Entrenamiento
- loop epochs
- early stopping (por valid loss)
- logging mínimo:
  - train_loss, valid_loss por epoch

---

## 7) Evaluación (OOS)
- inferencia sobre valid y test
- cálculo métricas ML:
  - MAE, RMSE, DA_last (y R2 opcional)
- cálculo métricas económicas (filtro):
  - Precision, Opportunity_Recall, Coverage

---

## 8) Guardado de artefactos (para Stage_08)
Guardar en una estructura estándar por modelo/horizonte:

- `reports/stage_07/<h>/<model_name>/metrics_ml.json`
- `reports/stage_07/<h>/<model_name>/metrics_econ.json`
- `reports/stage_07/<h>/<model_name>/pred_test.npz`
  - y_true, y_pred (test)
- `models/stage_07/<h>/<model_name>/model.*` (pesos / joblib / pt)

---

## 9) Resumen final
- tabla corta con métricas principales
- notas de entrenamiento (tiempo, convergencia, issues)